# Material de aula - Redes Neurais e Deep Learning


## <center> Laboratório 5 - Keras: Análise de Sentimento com LSTM: IMDB </center>

In [8]:
#CÉLULA KE-LIB-01
import numpy as np
import keras as k
import tensorflow as tf
import pandas as pd
import os
from matplotlib import pyplot as plt
import seaborn as sns
%matplotlib inline
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'

### Vamos construir uma rede recorrente do tipo LSTM para examinar dados textuais a fim de prever o sentimento (positivo ou negativo) de um comentário sobre um dado filme.

### A LSTM tem a seguinte topologia:

 - Uma camada de entrada do tipo Embedding capaz de converter cada palavra do texto em um vetor numerico de 32 valores.
 - A camada de memória (contexto) da rede possui 100 células.
 
 Dados adaptados de: http://ai.stanford.edu/~amaas/data/sentiment/

### 1 - Inicialização dos parâmetros. A inicialização fixa permite que os resultados possam ser reproduzidos de forma sistemática para melhorar as análises.

In [9]:
#CÉLULA KE-LIB-02
np.random.seed(1)
tf.set_random_seed(1)

### 2 - Carregamento do dataset e separação dos vetores X e Y

In [ ]:
#CÉLULA KE-LIB-03


### O dataset para esta atividade é composto por um arquivo de treinamento e um de teste, onde cada review possui no máximo, 50 palavras, no seguinte padrão:

    0 0 0 0 0 0 13 510 4 115 1331 363 . . . 1708 298 0 
    0 0 12 28 111 6 172 7 32188 9 4 88 31 . . . 1487 151 0

Cada linha constitui-se de uma opinião (review). Os valores iguais a zero no inicio de cada linha correspondem ao 'padding' a fim de manter todas as linhas com 50 valores.

O último valor em cada linha corresponde ao sentimento: 0 para negativo e 1 para positivo

Cada palavra foi codificada utilizando o mesmo padrão do dataset IMDB intrínseco ao Keras. Números de 0 a 3 possuem significado especial (no início da linha):

 - 0: padding, conforme apresentado anteriormente
 - 1: indica o inicio de um comentario onde os dados nao estão delimitados por quebra de linha
 - 2: indica o uso de palavras fora do vocabulário (OOV - Out Of Vocabulary), que não foram usadas no treinamento
 - 3: reservado para uso específico.
 
 As palavras foram convertidas para minúsculas, a pontuação foi removida exceto pelo sinal ' (apóstrofo) que é utilizado nos termos em inglês.
 
 O código identificador de cada palavra (token ID) é baseado na frequencia daquele token no dado de treinamento, onde 4 corresponde ao termo mais frequente "the", 5 é o segundo termo mais frequente "a" e assim por diante.
 
 O conjunto de treinamento possui 129.888 palavras distintas de forma que o ID do último token possui índice $I = 129.888 + 4 - 1 = 129.891 $
 
 **NOTA:** o conjunto de treinamento possui 620 linhas e o de teste possui 667 linhas

### 3 - Faça a separação dos dados em conjuntos de treinamento e teste

In [10]:
#CÉLULA KE-LIB-04
max_review_len=50

X_train = np.loadtxt("../input/imdb_train_50w.txt", delimiter=" ", usecols=range(0, max_review_len), dtype=np.float32)
y_train = np.loadtxt("../input/imdb_train_50w.txt", delimiter=" ", usecols=[max_review_len],         dtype=np.float32)
X_test  = np.loadtxt("../input/imdb_test_50w.txt",  delimiter=" ", usecols=range(0, max_review_len), dtype=np.float32)
y_test  = np.loadtxt("../input/imdb_test_50w.txt",  delimiter=" ", usecols=max_review_len,           dtype=np.float32)

### 4 - Monte o Modelo

In [15]:
#CÉLULA KE-LIB-05
#Montando a rede recorrente
tf.logging.set_verbosity(tf.logging.ERROR)

e_init = k.initializers.RandomUniform(-0.01, 0.01, seed=1)
init = k.initializers.glorot_uniform(seed=1)

theOptmizer = k.optimizers.Adam()
numOfwords = 129892
embed_vector_len = 32 #features para cada palavra do corpus

model = k.models.Sequential()

#word embedding
model.add(k.layers.embeddings.Embedding(input_dim=numOfwords, output_dim=embed_vector_len,
                                       embeddings_initializer=e_init, mask_zero=True))

model.add(k.layers.LSTM(units=100, kernel_initializer=init, dropout=0.2))
model.add(k.layers.Dense(units=1, kernel_initializer=init, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer=theOptmizer, metrics=['acc'])

model.summary()




_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_4 (Embedding)      (None, None, 32)          4156544   
_________________________________________________________________
lstm_4 (LSTM)                (None, 100)               53200     
_________________________________________________________________
dense_3 (Dense)              (None, 1)                 101       
Total params: 4,209,845
Trainable params: 4,209,845
Non-trainable params: 0
_________________________________________________________________


### 5 - Execute o Treinamento

In [16]:
#CÉLULA KE-LIB-06
#Treinamento
batch_size = 10
max_epochs = 5
print("Iniciando treinamento... ")
h = model.fit(X_train,y_train,batch_size=batch_size, epochs=max_epochs, shuffle=True, verbose=1)
print("Treinamento finalizado \n")

Iniciando treinamento... 
Epoch 1/5
620/620 [==============================] - 9s 14ms/step - loss: 0.6871 - acc: 0.5726
Epoch 2/5
620/620 [==============================] - 6s 9ms/step - loss: 0.5548 - acc: 0.7419
Epoch 3/5
620/620 [==============================] - 6s 9ms/step - loss: 0.2595 - acc: 0.9371
Epoch 4/5
620/620 [==============================] - 6s 9ms/step - loss: 0.0628 - acc: 0.9887
Epoch 5/5
620/620 [==============================] - 6s 9ms/step - loss: 0.1008 - acc: 0.9823
Treinamento finalizado 



### 6 - Faça a avaliaçao do Modelo

In [17]:
#CÉLULA KE-LIB-07
# Avaliação do modelo
eval = model.evaluate(X_test,y_test,verbose=0)
print("Dados de teste: loss = %0.6f accuracy = %0.2f%% \n" \
% (eval[0], eval[1]*100) )

Dados de teste: loss = 0.544890 accuracy = 74.21% 



### 7 - Salve o modelo em Arquivo

In [ ]:
# #CÉLULA KE-LIB-08
# # Salvando modelo em arquivo
# print("Salvando modelo em arquivo \n")
# mp = ".\\imdb_model.h5"
# model.save(mp)

### 8 - Faça um teste de operação com o Modelo

In [18]:
#CÉLULA KE-LIB-09
# 6. Usando modelo (operação)
print("usando o modelo para previsão de: {}".format('\"the movie was a great waste of my time\"'))
review = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
                    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 20, 16, 6, 86, 425, 7, 58, 64]], dtype=np.float32)

y_pred = model.predict(review)
print("Sentimento (0 = negativo, 1 = positivo) = {:0.4f}".format(y_pred[0][0]))#38% de ser positivo

usando o modelo para previsão de: "the movie was a great waste of my time"
Sentimento (0 = negativo, 1 = positivo) = 0.3895
